# Telco Customer Churn: Training Baselines

Trains a Logistic Regression and an XGBoost classifier on the churn data prepared in `explore_churn.ipynb`. Both models are frozen and saved for later use. The black box auditor in the next notebook will load these models and test how they degrade under noise.


## Imports

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

## Load the cleaned data

The clean version was saved to `data/processed` at the end of the exploration notebook. TotalCharges is already numeric here and the 11 empty rows are gone.

In [2]:
df_churn = pd.read_csv("../data/processed/churn_clean.csv")
print("Shape:", df_churn.shape)
print("TotalCharges NaNs:", df_churn["TotalCharges"].isna().sum())

Shape: (7032, 21)
TotalCharges NaNs: 0


## Split features and target

`customerID` is a unique identifier for each customer. It has no predictive value, so it is dropped before modeling.

The target column `Churn` is mapped from Yes/No to 1/0 so sklearn can use it.

In [3]:
df_clean = df_churn.drop("customerID", axis=1)

x = df_clean.drop("Churn", axis=1)
y = df_clean["Churn"].map({"Yes": 1, "No": 0})

print(x.shape, y.shape)

(7032, 19) (7032,)


## Train/test split

A stratified split keeps the same 73/27 class ratio in both halves. The fixed random state makes the split reproducible.

In [4]:
X_train, X_test, Y_train, Y_test = train_test_split(
    x, y, test_size=0.25, stratify=y, random_state=42
)

## One hot encoding

All text columns are converted to binary indicator columns. `drop_first=True` avoids the dummy variable trap by dropping one reference category per column. The split happens before encoding so the test set is never touched during training. Both halves are encoded with the same columns.

In [5]:
X_train = pd.get_dummies(X_train, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, drop_first=True, dtype=int)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5274, 30)
Test shape: (1758, 30)


## Logistic Regression baseline

`class_weight="balanced"` compensates for the 73/27 imbalance by weighting the minority class higher. The `liblinear` solver handles the unscaled numeric features without extra preprocessing.

In [6]:
model_lr = LogisticRegression(solver="liblinear", class_weight="balanced", random_state=42)
model_lr.fit(X_train, Y_train)

y_proba_lr = model_lr.predict_proba(X_test)[:, 1]
print(f"Logistic Regression ROC-AUC: {roc_auc_score(Y_test, y_proba_lr):.4f}")

Logistic Regression ROC-AUC: 0.8406


## XGBoost

A gradient boosted tree ensemble. The low learning rate with 45 trees gives it room to learn without overfitting. XGBoost does not need scaled features, so the same encoded data can be used as is.

In [7]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(n_estimators=45, learning_rate=0.05, random_state=42, objective="binary:logistic")
model_xgb.fit(X_train, Y_train)

y_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]
print(f"XGBoost ROC-AUC: {roc_auc_score(Y_test, y_proba_xgb):.4f}")

XGBoost ROC-AUC: 0.8399


## Freeze the models

Both models are saved with joblib. From here on they are treated as black boxes. The auditor only loads the file and calls `predict_proba`, nothing else. This is what "freezing a model" means for the rest of the project.

In [8]:
import joblib
import os

os.makedirs("../outputs/models", exist_ok=True)
joblib.dump(model_lr, "../outputs/models/churn_logistic_regression.joblib")
joblib.dump(model_xgb, "../outputs/models/churn_xgboost.joblib")

print("Models saved.")

Models saved.
